In [ ]:
pip install earthengine-api geemap

In [ ]:
import ee
import datetime

ee.Authenticate()
# Use default project or your personal account instead of ee-camcoredatabase
# If ee-camcoredatabase doesn't have access, comment out the project parameter:
try:
    ee.Initialize(project='ee-camcoredatabase')
    print("Using project: ee-camcoredatabase")
except Exception as e:
    print(f"Project initialization failed: {e}")
    print("Falling back to default initialization...")
    ee.Initialize()
    print("Using default GEE project")

# Brazil bounding box
bbox = ee.Geometry.BBox(-94.1875, -39.0208, 37.0625, 18.2292)

# ERA5-Land variables
variables = [
    "temperature_2m",
    "skin_temperature",
    "soil_temperature_level_1",
    "soil_temperature_level_2",
    "soil_temperature_level_3",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_pressure",
    "total_precipitation_sum",
    "surface_latent_heat_flux_sum",
    "surface_net_solar_radiation_sum",
    "evaporation_from_vegetation_transpiration_sum"
]

# ========== RESUME SETTINGS ==========
# TO RESUME: Set start_date to the day AFTER your last successful export
# Last successful: 2025-04-08, so resume from 2025-04-09
# For fresh start: Set to 2025-01-01

start_date = datetime.date(2025, 4, 9)  # ← CHANGE THIS TO RESUME

# ERA5-Land data availability: Currently up to 2025-08-04
# Update end_date as more data becomes available
end_date = datetime.date(2025, 8, 4)  # ← Latest available date

print("=" * 70)
print(f"ERA5-Land Export Configuration")
print("=" * 70)
print(f"Date range: {start_date} to {end_date}")
print(f"Total days to export: {(end_date - start_date).days + 1}")
print(f"Variables: {len(variables)}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")
print("\n⚠️  NOTE: ERA5-Land has ~2-3 month data lag")
print(f"    Already downloaded: Jan 1 - Apr 8, 2025 (98 days)")
print(f"    Resuming from: Apr 9, 2025")
print(f"    Current latest: {end_date}")
print("=" * 70)

In [ ]:
# Loop through each day in date range
current_date = start_date
exported_count = 0
total_days = (end_date - start_date).days + 1

print(f"\nStarting export from {start_date}...")
print(f"Progress updates every 30 days\n")

while current_date <= end_date:
    next_date = current_date + datetime.timedelta(days=1)
    date_str = current_date.strftime("%Y-%m-%d")
    next_date_str = next_date.strftime("%Y-%m-%d")

    # Filter dataset for this day
    dataset = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
               .filterBounds(bbox)
               .filterDate(date_str, next_date_str)
               .select(variables))

    # Get the image for this day
    image = dataset.mean().clip(bbox)

    # Export to Drive
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"ERA5_Brazil_{date_str}",
        folder="Brazil_ERA5_Daily",
        fileNamePrefix=f"ERA5_{date_str}",
        region=bbox,
        scale=10000,
        maxPixels=1e13
    )

    task.start()
    exported_count += 1
    
    # Progress updates
    if exported_count % 30 == 0:
        print(f"[{exported_count}/{total_days}] Exported through {date_str}")

    current_date = next_date

print("\n" + "=" * 70)
print(f"✓ All {exported_count} daily ERA5 exports initiated!")
print(f"Date range: {start_date} to {end_date}")
print("=" * 70)
print("\n📍 Check Google Earth Engine Tasks tab for progress")
print("📍 If interrupted again, update start_date in previous cell to resume")